# СС 8 + ЗС 8 — Есептер 1–3 (IoU, NMS, AP)
Студент: seed = 6

In [1]:
def area(b): return (b[2]-b[0])*(b[3]-b[1])

def iou(a, b, verbose=False):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = area(a) + area(b) - inter
    if verbose:
        print(f"  ∩ = {max(0,x2-x1)}×{max(0,y2-y1)} = {inter}; ∪ = {area(a)}+{area(b)}−{inter} = {union}")
    return inter / union

## Есеп 1 · IoU және зәкірлер
Объект `[40, 20, 100, 100]`, area = 60·80 = 4800.

RPN ережесі: IoU ≥ 0,7 → объект, IoU < 0,3 → фон, әйтпесе → еленбейді.

In [2]:
obj = [40, 20, 100, 100]
anchors = {"A1": [70,60,130,120], "A2": [30,30,90,90], "A3": [44,16,104,96]}
for n, a in anchors.items():
    v = iou(obj, a, verbose=True)
    lab = "объект (оң)" if v >= 0.7 else "фон" if v < 0.3 else "еленбейді"
    print(f"{n}: IoU = {v:.3f} → {lab}")

  ∩ = 30×40 = 1200; ∪ = 4800+3600−1200 = 7200
A1: IoU = 0.167 → фон
  ∩ = 50×60 = 3000; ∪ = 4800+3600−3000 = 5400
A2: IoU = 0.556 → еленбейді
  ∩ = 56×76 = 4256; ∪ = 4800+4800−4256 = 5344
A3: IoU = 0.796 → объект (оң)


**Қолмен шешім**

| Зәкір | ∩ | ∪ | IoU | Таңба |
|---|---|---|---|---|
| A1 `[70,60,130,120]` | 30×40 = 1200 | 4800+3600−1200 = 7200 | 0,167 | фон |
| A2 `[30,30,90,90]` | 50×60 = 3000 | 4800+3600−3000 = 5400 | 0,556 | еленбейді |
| A3 `[44,16,104,96]` | 56×76 = 4256 | 4800+4800−4256 = 5344 | 0,796 | объект |

## Есеп 2 · NMS (табалдырық 0,5)

In [3]:
boxes = [[20,20,100,100],[40,25,120,105],[60,30,140,110],[200,50,260,130],[205,55,265,135],[215,60,275,140]]
scores = [0.90, 0.86, 0.80, 0.84, 0.70, 0.60]

def nms(boxes, scores, thr, log=True):
    order = sorted(range(len(boxes)), key=lambda i: -scores[i])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        if log: print(f"  алдық №{i+1} (score {scores[i]})")
        rest = []
        for j in order:
            v = iou(boxes[i], boxes[j])
            if v > thr:
                if log: print(f"    №{j+1} жойылды: IoU(№{i+1},№{j+1}) = {v:.3f} > {thr}")
            else:
                rest.append(j)
        order = rest
    return [k+1 for k in sorted(keep)]

for thr in (0.5, 0.6):
    print(f"\nthr = {thr}")
    print("қалды:", nms(boxes, scores, thr))


thr = 0.5
  алдық №1 (score 0.9)
    №2 жойылды: IoU(№1,№2) = 0.542 > 0.5
  алдық №4 (score 0.84)
    №5 жойылды: IoU(№4,№5) = 0.753 > 0.5
  алдық №3 (score 0.8)
  алдық №6 (score 0.6)
қалды: [1, 3, 4, 6]

thr = 0.6
  алдық №1 (score 0.9)
  алдық №2 (score 0.86)
  алдық №4 (score 0.84)
    №5 жойылды: IoU(№4,№5) = 0.753 > 0.6
  алдық №3 (score 0.8)
  алдық №6 (score 0.6)
қалды: [1, 2, 3, 4, 6]


**Қолмен шешім (thr = 0,5).** Score бойынша реттеу: 1 (0,90), 2 (0,86), 4 (0,84), 3 (0,80), 5 (0,70), 6 (0,60).
1. №1 аламыз. IoU(1,2) = 0,54 > 0,5 → №2 жойылады; IoU(1,3) = 0,28 → №3 қалады.
2. №4 аламыз. IoU(4,5) = 0,75 > 0,5 → №5 жойылады; IoU(4,6) = 0,49 < 0,5 → №6 қалады.
3. №3 аламыз (№2 жойылған, №1-мен 0,28). №6 аламыз (№4-пен 0,49).

**Қалғаны: №1, №3, №4, №6.**

**ә) 0,6 табалдырықта:** IoU(1,2) = 0,54 < 0,6, сондықтан №2 енді жойылмайды; №5 әлі жойылады (0,75 > 0,6). Қалғаны: №1, №2, №3, №4, №6 — табалдырық жоғары болған сайын қайталанатын рамкалар көбірек қалады (FP көбейеді).

## Есеп 3 · AP

In [4]:
import numpy as np
tp = np.array([1,0,0,1,1,0,1])
n_gt = 5
ctp = np.cumsum(tp)
k = np.arange(1, len(tp)+1)
P = ctp / k
R = ctp / n_gt
for i in range(len(tp)):
    print(f"{i+1}  {'TP' if tp[i] else 'FP'}  P = {ctp[i]}/{k[i]} = {P[i]:.3f}   R = {ctp[i]}/{n_gt} = {R[i]:.2f}")

# қоршаушы (оң жақтағы max P)
env = np.maximum.accumulate(P[::-1])[::-1]
Rr = np.concatenate([[0], R])
envr = np.concatenate([[env[0]], env])
AP = np.sum((Rr[1:] - Rr[:-1]) * envr[1:])
print("қоршаушы:", np.round(env, 3))
print(f"AP = {AP:.4f}")

1  TP  P = 1/1 = 1.000   R = 1/5 = 0.20
2  FP  P = 1/2 = 0.500   R = 1/5 = 0.20
3  FP  P = 1/3 = 0.333   R = 1/5 = 0.20
4  TP  P = 2/4 = 0.500   R = 2/5 = 0.40
5  TP  P = 3/5 = 0.600   R = 3/5 = 0.60
6  FP  P = 3/6 = 0.500   R = 3/5 = 0.60
7  TP  P = 4/7 = 0.571   R = 4/5 = 0.80
қоршаушы: [1.    0.6   0.6   0.6   0.6   0.571 0.571]
AP = 0.5543


**Қолмен шешім**

| # | Score | TP/FP | Precision | Recall |
|---|---|---|---|---|
| 1 | 0,96 | TP | 1/1 = 1,000 | 0,20 |
| 2 | 0,91 | FP | 1/2 = 0,500 | 0,20 |
| 3 | 0,87 | FP | 1/3 = 0,333 | 0,20 |
| 4 | 0,80 | TP | 2/4 = 0,500 | 0,40 |
| 5 | 0,74 | TP | 3/5 = 0,600 | 0,60 |
| 6 | 0,62 | FP | 3/6 = 0,500 | 0,60 |
| 7 | 0,55 | TP | 4/7 = 0,571 | 0,80 |

Қоршаушы (әр recall-дан оңға қарай max P): R=0,2 → 1,0; R=0,4 → 0,6; R=0,6 → 0,6; R=0,8 → 0,571. R > 0,8 — детекция жоқ, үлес 0.

AP = 0,2·1,0 + 0,2·0,6 + 0,2·0,6 + 0,2·0,571 = 0,2 + 0,12 + 0,12 + 0,114 = **0,554**